In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Formal state-clock release template

This template intentionally has no published source binding. Bind a newly authorized immutable GitHub URL and 40-character SHA before Run all; do not substitute historical notebook SHAs for this candidate.


In [ ]:
from pathlib import Path
import subprocess, sys
RELEASE_SOURCE_URL = None
RELEASE_SOURCE_REF = None
if RELEASE_SOURCE_URL is None or RELEASE_SOURCE_REF is None:
    raise RuntimeError('Bind the authorized published GitHub URL and 40-character SHA before Run all.')
SOURCE = Path('/content/sc_sstw_formal_source')
if SOURCE.exists():
    if subprocess.check_output(['git', '-C', str(SOURCE), 'remote', 'get-url', 'origin'], text=True).strip() != RELEASE_SOURCE_URL:
        raise RuntimeError('unexpected origin')
else:
    subprocess.run(['git', 'init', str(SOURCE)], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', RELEASE_SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', RELEASE_SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
subprocess.run(['ffprobe', '-version'], check=True)
print('Pinned source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


## Fixed replication run

This binds the new formal source only after publication, uses the generated-terminal state-clock configuration, and writes a unique Drive result directory plus a continuously flushed launcher log. The runner persists its own progress and failures.


In [ ]:
from datetime import datetime, timezone
import os, signal
CONFIG = SOURCE / 'experiments/wan_state_clock/configs/generate_replication.json'
RUN_ID = 'wan_state_clock_formal_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/WanStateClockFormalReplication') / RUN_ID
if OUTPUT.exists():
    raise FileExistsError(OUTPUT)
cmd = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.run', '--config', str(CONFIG), '--output', str(OUTPUT)]
LOG = OUTPUT.parent / f'{RUN_ID}.launcher.log'
LOG.parent.mkdir(parents=True, exist_ok=True)
with LOG.open('w') as log:
    process = subprocess.Popen(cmd, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        code = process.wait()
    except BaseException:
        try:
            process.send_signal(signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
print('launcher exit', code)
print('Results:', OUTPUT)
print('Log:', LOG)
if code:
    raise subprocess.CalledProcessError(code, cmd)


In [ ]:
import json
result = json.loads((OUTPUT / 'result.json').read_text())
print(json.dumps({key: result[key] for key in ('status', 'source_commit', 'diagnostic_denominator', 'fixed_calls', 'actual_calls', 'failures') if key in result}, ensure_ascii=False, indent=2))
for name, row in result['videos'].items():
    print(name, row['status'], row.get('reporting_only'))
print('Full candidates/classes and observer traces:', OUTPUT / 'detections')
